In [16]:
import sys
import json
import torch
import numpy as np

from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from config import data_config
from config import preprocess_config

from preprocessing.preprocess_prices import preprocess_prices

In [17]:
closing_prices_dict = {}

for ticker in data_config.TICKERS:
    # Load JSON file
    with open(f'../data/raw/daily_data_{ticker}.json', 'r') as f:
        data = json.load(f)
    
    # Extract time series
    time_series = data['Time Series (Daily)']
    
    # Get closing prices sorted by date (oldest to newest)
    dates = sorted(time_series.keys())
    closing_prices = torch.tensor([float(time_series[date]['4. close']) for date in dates])
    
    # Store in dictionary
    closing_prices_dict[ticker] = closing_prices
    
    # Print summary
    print(f"{ticker}:")
    print(f"  {len(closing_prices)} closing prices")
    print(f"  Date range: {dates[0]} to {dates[-1]}")
    print(f"  Price range: ${closing_prices.min():.2f} to ${closing_prices.max():.2f}")
    print(f"  Array shape: {closing_prices.shape}")
    print()


NVDA:
  100 closing prices
  Date range: 2025-09-15 to 2026-02-05
  Price range: $170.29 to $207.04
  Array shape: torch.Size([100])

GOOG:
  100 closing prices
  Date range: 2025-09-15 to 2026-02-05
  Price range: $237.49 to $344.90
  Array shape: torch.Size([100])

AAPL:
  100 closing prices
  Date range: 2025-09-15 to 2026-02-05
  Price range: $236.70 to $286.19
  Array shape: torch.Size([100])



In [18]:
training_data = {}

# Loop through tickers
for ticker in data_config.TICKERS:
    closing_prices = closing_prices_dict[ticker]
    ticker_samples = []

    # Loop through T - len(window) + 1
    for start in range(len(closing_prices) - preprocess_config.T):
        returns, trend, realized_vol = preprocess_prices(closing_prices, start=start)
        
        # Sample dictionary
        sample = {
            "returns": returns.tolist(),
            "trend": trend.item(),
            "realized_vol": realized_vol.item()
        }
        ticker_samples.append(sample)
    
    # Store samples
    training_data[ticker] = ticker_samples
    
    # Save to JSON file
    output_path = f'../data/preprocessed/train_data_{ticker}.json'
    with open(output_path, 'w') as f:
        json.dump(ticker_samples, f, indent=2)
    
    print(f"{ticker}: {len(ticker_samples)} samples saved to {output_path}")

print(f"\nTotal samples across all tickers: {sum(len(samples) for samples in training_data.values())}")


NVDA: 36 samples saved to ../data/preprocessed/train_data_NVDA.json
GOOG: 36 samples saved to ../data/preprocessed/train_data_GOOG.json
AAPL: 36 samples saved to ../data/preprocessed/train_data_AAPL.json

Total samples across all tickers: 108


In [20]:
# Load and combine all preprocessed data 
all_samples = []
for ticker in data_config.TICKERS:
    all_samples.extend(training_data[ticker])

print(f"Total samples before upsampling: {len(all_samples)}")

# Absolute trends
abs_trends = [abs(sample['trend']) for sample in all_samples]

# Percentiles (75th, 90th, 95th)
threshold_25 = np.percentile(abs_trends, 75)
threshold_10 = np.percentile(abs_trends, 90)
threshold_5 = np.percentile(abs_trends, 95)

# Apply upsampling 
upsampled_data = []

for i in range(len(all_samples)):
    sample = all_samples[i]
    abs_trend = abs_trends[i]
    
    # Replicate according to tier
    if abs_trend >= threshold_5:
        for _ in range(5):
            upsampled_data.append(sample)
    elif abs_trend >= threshold_10:
        for _ in range(5):
            upsampled_data.append(sample)
    elif abs_trend >= threshold_25:
        for _ in range(5):
            upsampled_data.append(sample)
    else:
        upsampled_data.append(sample)

# Save to single JSON file
output_path = '../data/upsampled/upsampled_data.json'
with open(output_path, 'w') as f:
    json.dump(upsampled_data, f, indent=2)

print(f"\nSaved {len(upsampled_data)} samples to {output_path}")
print(f"Upsampled {len(upsampled_data) - len(all_samples)} samples")

Total samples before upsampling: 108

Saved 216 samples to ../data/upsampled/upsampled_data.json
Upsampled 108 samples
